# Building the volume-surge dataset

Build `features.parquet` from hourly trades and market metadata. Each output row contains 28 features and a label for one market at one prediction time.

Run this notebook before `analysis.ipynb`. All timestamps are UTC.


## 0. Load the data and helper functions

Read hourly trades from `hp` in `data/volume_panel.duckdb`. Load `markets` and `resolutions` from `data/catalog/`. Use `markets` for market context and `resolutions` for closure filtering. Open the source database read-only and build intermediate tables in a temporary connection.


In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

LITE_DIR = Path.cwd().resolve()
if not (LITE_DIR / "scripts" / "features.py").exists():
    LITE_DIR = LITE_DIR / "lite"

CATALOG_DIR = LITE_DIR / "data" / "catalog"
DATABASE_PATH = LITE_DIR / "data" / "volume_panel.duckdb"
OUTPUT_PATH = LITE_DIR / "features.parquet"

sys.path.insert(0, str(LITE_DIR / "scripts"))
from features import engineer_features

DATA_END = pd.Timestamp("2026-10-26", tz="UTC")  # Exclusive cutoff.
MIN_MARKET_VOLUME = 20_000
MIN_MARKET_FILLS = 200
DECISION_STRIDE_HOURS = 4


In [2]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")
con.execute("SET preserve_insertion_order = false")
# DuckDB raises an IO error instead of creating a missing spill directory.
SPILL_DIR = Path("/tmp/lite_feature_build_spill")
SPILL_DIR.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory = '{SPILL_DIR}'")

# Escape the path as an SQL string; no source tables are modified.
database_sql_path = str(DATABASE_PATH).replace("'", "''")
con.execute(f"ATTACH '{database_sql_path}' AS source (READ_ONLY)")
con.execute("CREATE TEMP VIEW hp AS SELECT * FROM source.hp")

for table_name in ["markets", "resolutions"]:
    catalog_path = CATALOG_DIR / f"{table_name}.parquet"
    con.execute(
        f"CREATE TEMP TABLE {table_name} AS SELECT * FROM read_parquet(?)",
        [str(catalog_path)],
    )


def query(sql, parameters=None):
    """Return a query result as a pandas table."""
    return con.execute(sql, parameters).df()


### Input fields

Each `hp` row represents one market in one hourly bucket:

| Field | Meaning |
|---|---|
| `vol` | Total fill amount in USD. |
| `fills` | Number of fill records. |
| `wallets` | Number of distinct wallets within the hour. |
| `p_last` | Last recorded price in the hour. |

Keep the supplied price convention; normalization across outcomes is unspecified. Use catalog `resolved_at` for closure filtering; it may fall back to the scheduled end date.

## 1. Select the market sample

Keep markets with at least **$20,000 in volume** and **200 fills** over the full observed period.

Rebuild market summaries from `hp`. This is a retrospective sample: selection uses activity across the entire period, including activity after individual prediction times.

In [3]:
con.execute("""
    CREATE TEMP TABLE market_summary AS
    SELECT
        market,
        sum(vol) AS total_d,
        sum(fills) AS total_fills,
        count(*) AS active_hours,
        min(h) AS first_h,
        max(h) AS last_h
    FROM hp
    GROUP BY market
""")

con.execute(f"""
    CREATE TEMP TABLE universe AS
    SELECT *
    FROM market_summary
    WHERE total_d >= {MIN_MARKET_VOLUME}
      AND total_fills >= {MIN_MARKET_FILLS}
""");

## 2. Mark source coverage

Build an hourly calendar for the source period. Mark an hour as observed if any market has a record; otherwise, mark coverage as unknown.


In [4]:
con.execute("""
    CREATE TEMP TABLE coverage AS
    SELECT
        calendar.h,
        observed.h IS NOT NULL AS source_observed
    FROM generate_series(
        TIMESTAMP '2026-05-30',
        TIMESTAMP '2026-10-25 23:00:00',
        INTERVAL '1 hour'
    ) AS calendar(h)
    LEFT JOIN (
        SELECT DISTINCT h
        FROM hp
    ) AS observed USING (h)
""");

## 3. Build a continuous hourly grid

Add missing market-hour rows so rolling windows measure consecutive hours.

For each selected market, build the grid from **71 hours before its first trade bucket** to **35 hours after its last trade bucket**, capped at the data cutoff:

- The 71 preceding hours plus the first trade bucket provide a 72-hour history.
- The next 23 hours can still include the last trade in a 24-hour volume window.
- The final 12 hours provide outcomes for those candidate predictions.

Fill missing volume and count fields with zero, and retain these coverage flags:

| Flag | Meaning |
|---|---|
| `market_hour_present` | The market has a row in `hp` for this hour. |
| `source_observed` | At least one market has a source record for this hour. |
| `hour_known` | The source is observed, or the entire hour ended before market creation. |

Zeros in unknown hours are placeholders. Later steps remove predictions whose required windows contain these hours.

In [5]:
con.execute("""
    CREATE TEMP TABLE hourly AS
    WITH grid AS (
        SELECT
            market,
            unnest(generate_series(
                first_h - INTERVAL '71 hour',
                least(
                    last_h + INTERVAL '35 hour',
                    TIMESTAMP '2026-10-25 23:00:00'
                ),
                INTERVAL '1 hour'
            )) AS h
        FROM universe
    )
    SELECT
        grid.market,
        grid.h,
        coalesce(trades.vol, 0)::DOUBLE AS vol,
        coalesce(trades.fills, 0)::DOUBLE AS fills,
        coalesce(trades.wallets, 0)::DOUBLE AS wallets,
        coalesce(trades.buy_d, 0)::DOUBLE AS buy_d,
        trades.p_last AS praw,
        trades.market IS NOT NULL AS market_hour_present,
        coalesce(coverage.source_observed, false) AS source_observed,
        (
            coalesce(coverage.source_observed, false)
            OR coalesce(
                grid.h + INTERVAL '1 hour' <= CAST(markets.created_at AS TIMESTAMP),
                false
            )
        ) AS hour_known
    FROM grid
    LEFT JOIN hp AS trades
        ON trades.market = grid.market AND trades.h = grid.h
    LEFT JOIN coverage ON coverage.h = grid.h
    LEFT JOIN markets ON markets.slug = grid.market
""");

## 4. Carry prices forward

Set `pf` to the latest recorded price available at each bucket's close. Carry prices forward through observed hours with no trades.

An unknown hour breaks the price history. Leave `pf` missing until a new price is recorded. Never fill prices backward.

In [6]:
con.execute("""
    CREATE TEMP TABLE priced AS
    WITH price_history AS (
        SELECT
            *,
            last_value(praw IGNORE NULLS) OVER history AS carried_price,
            max(CASE WHEN praw IS NOT NULL THEN h END) OVER history AS last_price_h,
            max(CASE WHEN NOT hour_known THEN h END) OVER history AS last_unknown_h
        FROM hourly
        WINDOW history AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    )
    SELECT
        *,
        CASE
            WHEN last_unknown_h IS NULL OR last_price_h > last_unknown_h
            THEN carried_price
            ELSE NULL
        END AS pf
    FROM price_history
""");

## 5. Calculate rolling windows

For `h = 12:00`, the bucket is `[12:00, 13:00)` and prediction occurs at `13:00`.

| Quantity | Buckets included |
|---|---|
| `vol6` | 07:00 through 12:00. |
| `vol24` | Yesterday at 13:00 through today at 12:00. |
| `vol72` | The 72 buckets ending at 12:00. |
| `vol6_prev` | 01:00 through 06:00. |
| `vol24_prev` | The 24 buckets immediately before the current 24-hour window. |
| `vol_fwd12` | 13:00 through tomorrow at 00:00; the final bucket closes at 01:00. |

Calculate windows separately for each market. Count known hours in the 72-hour history and 12-hour outcome window.

`wallets24` sums hourly distinct-wallet counts. The price range uses hourly closing/carried prices. Cumulative volume starts at the source observation boundary.

In [7]:
con.execute("""
    CREATE TEMP TABLE windowed AS
    SELECT
        market,
        h,
        vol,
        pf,
        sum(vol) OVER past6 AS vol6,
        sum(vol) OVER past24 AS vol24,
        sum(vol) OVER past72 AS vol72,
        sum(fills) OVER past24 AS fills24,
        sum(wallets) OVER past24 AS wallets24,
        sum(buy_d) OVER past24 AS buy24,
        max(vol) OVER past24 AS vmax24,
        sum(vol) OVER previous6 AS vol6_prev,
        sum(vol) OVER previous24 AS vol24_prev,
        sum(vol) OVER cumulative AS cum_vol,
        max(pf) OVER past24 AS pmax24,
        min(pf) OVER past24 AS pmin24,
        lag(pf, 1) OVER chronological AS pf_1,
        lag(pf, 6) OVER chronological AS pf_6,
        lag(pf, 24) OVER chronological AS pf_24,
        sum(vol) OVER future12 AS vol_fwd12,
        sum(CAST(hour_known AS INTEGER)) OVER past72 AS history_known_hours,
        sum(CAST(hour_known AS INTEGER)) OVER future12 AS future_known_hours
    FROM priced
    WINDOW
        chronological AS (PARTITION BY market ORDER BY h),
        past6 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ),
        past24 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        ),
        past72 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 71 PRECEDING AND CURRENT ROW
        ),
        previous6 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 11 PRECEDING AND 6 PRECEDING
        ),
        previous24 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 47 PRECEDING AND 24 PRECEDING
        ),
        cumulative AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        future12 AS (
            PARTITION BY market ORDER BY h
            ROWS BETWEEN 1 FOLLOWING AND 12 FOLLOWING
        )
""");

## 6. Sample prediction times and join market context

Keep bucket hours **00, 04, 08, 12, 16, and 20**. Predictions occur one hour later, at bucket close. Require at least **$100 in past-24-hour volume** and a bucket at least **two hours after the first observed trade**.

Join market metadata and closure times. Measure `age_h` from the first observed trade to `decision_time`. Use `decision_time` for calendar features, time to planned end, closure filtering, and later data splits. Save `created_age_h` for inspection.

In [8]:
con.execute(fr"""
    CREATE TEMP TABLE candidates AS
    SELECT
        windows.*,
        windows.h + INTERVAL '1 hour' AS decision_time,
        markets.event_slug,
        markets.neg_risk,
        CAST(markets.created_at AS TIMESTAMP) AS created_at,
        CAST(markets.end_date AS TIMESTAMP) AS planned_end,
        epoch(
            (windows.h + INTERVAL '1 hour') - universe.first_h
        ) / 3600.0 AS age_h,
        epoch(
            (windows.h + INTERVAL '1 hour') - CAST(markets.created_at AS TIMESTAMP)
        ) / 3600.0 AS created_age_h,
        epoch(
            CAST(markets.end_date AS TIMESTAMP) - (windows.h + INTERVAL '1 hour')
        ) / 3600.0 AS hours_to_end,
        CAST(resolutions.resolved_at AS TIMESTAMP) AS resolved_at
    FROM windowed AS windows
    JOIN universe USING (market)
    LEFT JOIN markets ON markets.slug = windows.market
    LEFT JOIN resolutions ON resolutions.market_slug = windows.market
    WHERE windows.vol24 >= 100
      AND hour(windows.h) % {DECISION_STRIDE_HOURS} = 0
      AND windows.h >= universe.first_h + INTERVAL '2 hour'
""")

candidates = query("SELECT * FROM candidates ORDER BY h, market")
time_columns = ["h", "decision_time", "created_at", "planned_end", "resolved_at"]
for column in time_columns:
    candidates[column] = pd.to_datetime(candidates[column], utc=True)


## 7. Flag incomplete windows and market closures

Require **72 known historical hours** and **12 known future hours**, with the outcome window ending by the data cutoff. Entire hours before confirmed market creation count as known zeros.

Exclude candidates with `resolved_at <= decision_time`: these markets are already closed. Keep candidates with missing closure times.

Also exclude `decision_time < resolved_at <= decision_time + 12 hours`, retaining the existing retrospective pre-closure rule.


In [9]:
outcome_end = candidates.decision_time + pd.Timedelta(hours=12)

candidates["history_complete"] = candidates.history_known_hours.eq(72)
candidates["future_complete"] = (
    candidates.future_known_hours.eq(12)
    & outcome_end.le(DATA_END)
)
candidates["already_closed"] = candidates.resolved_at.le(candidates.decision_time)
candidates["near_resolution"] = (
    candidates.resolved_at.gt(candidates.decision_time)
    & candidates.resolved_at.le(outcome_end)
)


## 8. Create the features and label

Keep candidates with complete historical windows and exclude already-closed markets. Apply `engineer_features()` from `scripts/features.py` to these candidates, so closed markets do not contribute to peer activity. Compute peer activity before filtering on future coverage or upcoming closure.

Set **`y = 1`** when future 12-hour volume is **at least $10,000** and **greater than past 24-hour volume**; otherwise, set `y = 0`.

Keep rows with complete future windows and no closure in the next 12 hours. Future volume and closure time are excluded from model inputs.

In [10]:
history_complete = candidates.loc[candidates.history_complete].copy()
still_open = history_complete.loc[~history_complete.already_closed].copy()

# Derive peer activity before using future information to filter the sample.
engineered = engineer_features(still_open)
keep = engineered.future_complete & ~engineered.near_resolution
features_df = engineered.loc[keep].copy()
features_df["pipeline_version"] = "lite-v3"
features_df = features_df.sort_values(
    ["decision_time", "market"]
).reset_index(drop=True)


### Define the 28 model inputs

Use four feature blocks: activity (A), acceleration (B), price (C), and context (D). Here, `log1p(x) = ln(1 + x)`.

| Block | Inputs | Definition |
|---|---|---|
| A | `log_vol6`, `log_vol24`, `log_vol72` | `log1p` of each volume. |
| A | `fills24` | Fill count over 24 hours. |
| A | `log_wallets24` | `log1p` of summed hourly distinct-wallet counts. |
| A | `buy_share24` | `buy24 / (vol24 + 1)`; BUY-side amount, not necessarily YES buying. |
| A | `avg_fill24` | `vol24 / fills24`; missing if `fills24 <= 0`. |
| A | `peak_share24` | `vmax24 / vol24`; missing if `vol24 <= 0`. |
| A | `log_cum_vol` | `log1p(cum_vol)` since the observation boundary. |
| B | `share6`, `share1` | `vol6 / (vol24 + 1)` and `vol / (vol24 + 1)`. |
| B | `growth6`, `growth24` | `vol6 / (vol6_prev + 1)` and `vol24 / (vol24_prev + 1)`. |
| C | `pf` | Latest available price at bucket close. |
| C | `d_pf_1`, `d_pf_6`, `d_pf_24` | `pf` minus its value 1, 6, or 24 hours earlier. |
| C | `range24` | `pmax24 - pmin24`; fill missing values with 0. |
| C | `dist_bound` | `min(pf, 1 - pf)`; fill missing values with 0.5. |
| D | `hour_sin`, `hour_cos` | Sine/cosine of `2π × decision hour / 24`. |
| D | `dow_sin`, `dow_cos` | Sine/cosine of `2π × decision weekday / 7`; Monday = 0. |
| D | `log_age` | `log1p(max(age_h, 0))`. |
| D | `log_h2end` | `log1p(hours_to_end)` after clipping to `[0, 8760]`. |
| D | `h2end_cap` | `hours_to_end` clipped to `[-72, 2160]`. |
| D | `neg_risk` | Catalog market-structure flag; fill missing values with 0. |
| D | `sib_accel` | Mean `accel6` of other eligible markets in the same event and hour; missing without peers. |

`accel6 = (vol6 / 6) / (vol24 / 24 + 1)` supports the baseline and peer feature. Neither `accel6` nor `created_age_h` is among the 28 model inputs.


## 9. Save the dataset

Write `features.parquet`.


In [11]:
features_df.to_parquet(OUTPUT_PATH, index=False)